# Task 09 — CatBoost loss and hyperparameter selection

Этот notebook читает только опубликованный artifact. Selection выполняется на `rolling_1 -> rolling_2` и `rolling_2 -> rolling_3`; основной критерий — средний `precision_at_20_labeled_users`. Canonical fold открывается ровно один раз после фиксации winner.

In [ ]:
from pathlib import Path
import json
import polars as pl

artifact = Path("../artifacts/task09_catboost_selection_v1")
if not artifact.is_dir():
    raise FileNotFoundError(
        "Task 09 full artifact is not published yet; run the production launcher first."
    )
config = json.loads((artifact / "config.json").read_text())
metrics = json.loads((artifact / "metrics.json").read_text())
winner = json.loads((artifact / "selection" / "winner.json").read_text())
leaderboard = pl.read_parquet(artifact / "selection" / "leaderboard.parquet")

In [ ]:
summary = pl.DataFrame(
    {
        "winner_config_id": [metrics["winner_config_id"]],
        "rolling_mean_p20_labeled": [
            metrics["selection"]["mean_precision_at_20_labeled_users"]
        ],
        "canonical_p20_labeled": [metrics["precision_at_20_labeled_users"]],
        "canonical_p20_all": [metrics["precision_at_20_all_targets"]],
        "canonical_hits": [metrics["final_hits"]],
        "runtime_seconds": [metrics["runtime_seconds"]],
        "peak_memory_mb": [metrics["peak_memory_mb"]],
    }
)
summary

In [ ]:
leaderboard.sort(
    [
        "mean_precision_at_20_labeled_users",
        "min_precision_at_20_labeled_users",
        "fold_spread_precision_at_20_labeled_users",
    ],
    descending=[True, True, False],
)

In [ ]:
fold_rows = []
for result in metrics["selection"]["fold_results"]:
    fold_rows.append(
        {
            "pair_id": result["pair_id"],
            "train_fold": result["train_fold"],
            "eval_fold": result["eval_fold"],
            "precision_at_20_labeled_users": result["precision_at_20_labeled_users"],
            "precision_at_20_all_targets": result["precision_at_20_all_targets"],
            "final_hits": result["final_hits"],
            "tree_count": result["tree_count"],
        }
    )
pl.DataFrame(fold_rows)

In [ ]:
pl.DataFrame(
    {
        "comparison": ["Task 08", "RRF full", "Task 09 winner"],
        "precision_at_20_labeled_users": [
            metrics["comparisons"]["task08"]["precision_at_20_labeled_users"],
            metrics["comparisons"]["rrf_full"]["precision_at_20_labeled_users"],
            metrics["precision_at_20_labeled_users"],
        ],
        "precision_at_20_all_targets": [
            metrics["comparisons"]["task08"]["precision_at_20_all_targets"],
            metrics["comparisons"]["rrf_full"]["precision_at_20_all_targets"],
            metrics["precision_at_20_all_targets"],
        ],
    }
)